# H-014 · Adaptive z-window (trad-z)

Arms: fixed (`Z_WINDOW_STAR` from H-003, default 60) vs `clip(2 * HL_{t-1}, 20, 120)` vs alt `[10, 252]`. Not an H-008 trade/no-trade gate.

Type `Z_WINDOW_MODE_STAR` as `"fixed"`, `"adaptive"`, or `"adaptive_alt"`.

**Pinch of salt:** fixed z-window length was already screened under `BREAK_STAR` in H-005 (`Z_WINDOW_STAR` in {40,60,90}). Treat this adaptive bake-off cautiously — possible overfit vs that look.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    fold_table,
    fold_val_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    ARTIFACTS_DIR,
    DEFAULT_STAR_STACK,
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    frozen_pairs_for_universe,
    load_universe_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    overlay_ols_hedge,
    repo_root,
    split_is_oos,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds
from strategies.s2_coint.config import S2SimConfig

STAR_PATH = DEFAULT_STAR_STACK
TEARSHEET_DIR = ARTIFACTS_DIR
stack = load_star_stack(STAR_PATH)
# Universe from H-001. PAIRS_STAR is only populated when BOOK_STAR = freeze (H-004);
# otherwise fall back to the ranked frozen book written by H-001.
require_star("UNIVERSE_STAR", stack.get("UNIVERSE_STAR"))
UNIVERSE = str(stack["UNIVERSE_STAR"])
PAIRS_STAR = list(stack.get("PAIRS_STAR") or frozen_pairs_for_universe(UNIVERSE, "1d", root=ROOT))
print("stack keys:", sorted(stack))
print("UNIVERSE", UNIVERSE, "PAIRS", PAIRS_STAR)


## 1. Load frozen panel / PAIRS_STAR


In [ ]:
require_star("BAR_STAR", stack.get("BAR_STAR"))
BAR = str(stack["BAR_STAR"])
train, full = load_universe_panels(UNIVERSE, BAR, PAIRS_STAR, root=ROOT)
lb = lookbacks_for_bar(
    BAR,
    ols_days=int(stack["OLS_WINDOW_STAR"]) if stack.get("OLS_WINDOW_STAR") not in (None, "") else None,
    z_days=int(stack["Z_WINDOW_STAR"]) if stack.get("Z_WINDOW_STAR") not in (None, "") else None,
    adf_days=int(stack["ADF_WINDOW_STAR"]) if stack.get("ADF_WINDOW_STAR") not in (None, "") else None,
)
train = overlay_ols_hedge(
    train,
    ols_window=lb["ols_window"],
    z_window=lb["z_window"],
    hl_window=lb["hl_window"],
    adf_window=lb["adf_window"],
)
full = overlay_ols_hedge(
    full,
    ols_window=lb["ols_window"],
    z_window=lb["z_window"],
    hl_window=lb["hl_window"],
    adf_window=lb["adf_window"],
)
if stack.get("HEDGE_STAR") == "kalman":
    kf_delta = stack["KALMAN_DELTA_STAR"]
    train = overlay_kalman_hedge(
        train,
        burn_in=lb["kalman_burn_in"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
        delta=kf_delta,
    )
    full = overlay_kalman_hedge(
        full,
        burn_in=lb["kalman_burn_in"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
        delta=kf_delta,
    )

is_end = is_end_for_stack(stack, full if BAR == "1h" else train)
if BAR == "1d":
    is_panel, oos_panel = train.copy(), full.loc[pd.to_datetime(full["date"]) > is_end].copy()
else:
    is_panel, oos_panel = split_is_oos(full, is_end=is_end)
s1_weekly = load_s1_weekly(ROOT)
print("bar", BAR, "is_end", is_end, "IS rows", len(is_panel), "OOS rows", len(oos_panel))
is_panel.head()


## 2. Attach this-hyp columns


In [ ]:
panel = is_panel.copy()
print(panel.columns.tolist())
panel.head()


## 3. Walk-forward folds


In [ ]:
dates = pd.DatetimeIndex(pd.to_datetime(panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
fold_table(folds)


## 4. Fold-val metrics (validation only)


In [ ]:
base = config_from_stack(stack)
configs = {
    "fixed": config_from_stack(stack, z_window_mode="fixed"),
    "adaptive": config_from_stack(stack, z_window_mode="adaptive"),
    "adaptive_alt": config_from_stack(stack, z_window_mode="adaptive_alt"),
}
fold_df = fold_val_metrics(panel, folds, configs, s1_weekly=s1_weekly)
fold_df


## 5. Boxplots (do not assign STAR here)


In [ ]:
plot_fold_boxplots(fold_df, title="H-014 fold-val")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
fold_df.groupby("arm")[["ann_sharpe", "max_drawdown", "corr_to_s1"]].median()


## 6. Type `Z_WINDOW_MODE_STAR` then save


In [ ]:
Z_WINDOW_MODE_STAR = None  # TODO set after review — do not use argmax / median Sharpe
require_star("Z_WINDOW_MODE_STAR", Z_WINDOW_MODE_STAR)
stack["Z_WINDOW_MODE_STAR"] = Z_WINDOW_MODE_STAR
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH)


## 7. Sealed OOS once


In [ ]:
require_star("Z_WINDOW_MODE_STAR", Z_WINDOW_MODE_STAR)
oos_cfg = config_from_stack(load_star_stack(STAR_PATH))
oos = run_s2_backtest(oos_panel, oos_cfg, s1_weekly=s1_weekly)
print(oos.metrics)
write_tearsheet_pdf(tearsheet_path("H-014", str(Z_WINDOW_MODE_STAR)), oos.returns, title="H-014 sealed OOS")
print("tearsheet", tearsheet_path("H-014", str(Z_WINDOW_MODE_STAR)))


## 8. Notes for next hyp


H-015 entry extremity scores is next (trad_z | v1 roll/ewm | v2 OU | v3 HMM). No copula.
